# Cross-Encoder Few-Shot Generalization Experiment

This notebook evaluates whether the cross-encoder grounding model transfers to novel domains.

## Experimental Setup

| Setting | Description |
|---------|-------------|
| **Zero-shot** | Pre-trained cross-encoder evaluated directly on LIBERO |
| **Few-shot (N)** | Pre-trained cross-encoder fine-tuned on N LIBERO examples |
| **From Scratch (N)** | Random-init BERT fine-tuned on same N LIBERO examples |

## Key Result from Zero-Shot Eval

| Model | LIBERO Zero-Shot Accuracy |
|-------|---------------------------|
| Prefix Model | 9.5% |
| **Cross-Encoder** | **42.25%** |

The cross-encoder already shows strong transfer. This experiment tests if few-shot fine-tuning can close the remaining gap.

In [ ]:
# Install dependencies
import sys, subprocess, pkgutil
def _pip(pkg): 
    if pkgutil.find_loader(pkg.split("==")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("transformers")
_pip("scikit-learn")
_pip("pandas")
_pip("accelerate")

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from copy import deepcopy
from collections import defaultdict
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Paths
PRETRAINED_MODEL_DIR = Path("outputs_cross_encoder/final")  # Pre-trained cross-encoder
LIBERO_DATA_DIR = Path("libero_grounding_data")  # LIBERO grounding data
OUTPUT_DIR = Path("outputs_cross_encoder_fewshot")
OUTPUT_DIR.mkdir(exist_ok=True)

# LIBERO suites to use
SUITES = ["libero_10", "libero_90", "libero_object", "libero_goal", "libero_spatial"]

# Few-shot settings (number of LIBERO *groups* to fine-tune on)
# Each group has 1 predicate + N arguments, so actual pairs will be more
FEWSHOT_SIZES = [25, 50, 100, 200]

# Model config
BASE_MODEL_NAME = "bert-base-uncased"
INCLUDE_CONTEXT = True  # Include domain/type in candidate strings

# Training hyperparameters
FINETUNING_CONFIG = {
    "num_epochs": 5,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "neg_ratio": 3,  # Negative samples per positive
    "eval_batch_size": 64,
}

# Test split ratio
TEST_RATIO = 0.3

print(f"Pre-trained model: {PRETRAINED_MODEL_DIR}")
print(f"LIBERO data: {LIBERO_DATA_DIR}")
print(f"Few-shot sizes: {FEWSHOT_SIZES}")
print(f"Test ratio: {TEST_RATIO}")

## 1. Model Definition

In [ ]:
class CrossEncoderForGrounding(nn.Module):
    """Cross-Encoder model for grounding."""
    
    def __init__(self, config):
        super().__init__()
        self.bert = AutoModel.from_config(config)
        self.classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1),
        )
    
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output).squeeze(-1)
        return {'logits': logits}


def load_pretrained_cross_encoder(model_dir: Path, device: torch.device):
    """Load pre-trained cross-encoder model."""
    config = AutoConfig.from_pretrained(model_dir)
    model = CrossEncoderForGrounding(config)
    
    # Load weights
    safetensors_path = model_dir / "model.safetensors"
    pytorch_path = model_dir / "pytorch_model.bin"
    
    if safetensors_path.exists():
        from safetensors.torch import load_file
        state_dict = load_file(safetensors_path)
    elif pytorch_path.exists():
        state_dict = torch.load(pytorch_path, map_location='cpu', weights_only=False)
    else:
        raise FileNotFoundError(f"No model weights found in {model_dir}")
    
    model.load_state_dict(state_dict)
    model.to(device)
    return model


def create_cross_encoder_from_scratch(device: torch.device):
    """Create a cross-encoder with random initialization."""
    config = AutoConfig.from_pretrained(BASE_MODEL_NAME)
    model = CrossEncoderForGrounding(config)
    model.to(device)
    return model


def copy_model(model: CrossEncoderForGrounding, device: torch.device) -> CrossEncoderForGrounding:
    """Create a deep copy of a model."""
    config = model.bert.config
    new_model = CrossEncoderForGrounding(config)
    new_model.load_state_dict(deepcopy(model.state_dict()))
    new_model.to(device)
    return new_model

## 2. Data Loading and Processing

In [ ]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load JSONL file."""
    rows = []
    if not path.exists():
        print(f"[warn] File not found: {path}")
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_libero_data(data_dir: Path, suites: List[str]) -> List[Dict[str, Any]]:
    """Load LIBERO grounding data from multiple suites."""
    all_rows = []
    for suite in suites:
        path = data_dir / f"{suite}_grounding.jsonl"
        rows = load_jsonl(path)
        for r in rows:
            if 'suite' not in r:
                r['suite'] = suite
        print(f"  {suite}: {len(rows)} entries")
        all_rows.extend(rows)
    return all_rows


def split_by_task(data: List[Dict], test_ratio: float = 0.3) -> Tuple[List[Dict], List[Dict]]:
    """
    Split data by task_name to avoid data leakage.
    All entries from the same task go to either train or test.
    """
    task_to_entries = defaultdict(list)
    for entry in data:
        task_name = entry.get('task_name', entry.get('id', 'unknown'))
        task_to_entries[task_name].append(entry)
    
    tasks = list(task_to_entries.keys())
    random.shuffle(tasks)
    
    n_test = int(len(tasks) * test_ratio)
    test_tasks = set(tasks[:n_test])
    train_tasks = set(tasks[n_test:])
    
    train_data = [e for t in train_tasks for e in task_to_entries[t]]
    test_data = [e for t in test_tasks for e in task_to_entries[t]]
    
    return train_data, test_data


# Load LIBERO data
print(f"Loading LIBERO data from: {LIBERO_DATA_DIR}")
all_libero_data = load_libero_data(LIBERO_DATA_DIR, SUITES)
print(f"\nTotal LIBERO entries: {len(all_libero_data)}")

# Split into train pool and held-out test set
libero_train_pool, libero_test = split_by_task(all_libero_data, test_ratio=TEST_RATIO)
print(f"\nTrain pool: {len(libero_train_pool)} entries")
print(f"Test set (held out): {len(libero_test)} entries")

In [ ]:
def get_candidates_from_prefix(entry: Dict) -> List[str]:
    """
    Extract candidate list from prefix.
    For predicates: tokens after <predicates>
    For arguments: tokens after <const>
    """
    prefix = entry.get('prefix', [])
    grounding_type = entry.get('grounding_type', '')
    
    if grounding_type == 'predicate':
        if '<predicates>' in prefix:
            idx = prefix.index('<predicates>')
            return prefix[idx + 1:]
    else:  # argument
        if '<const>' in prefix:
            idx = prefix.index('<const>')
            return prefix[idx + 1:]
    
    return []


def get_domain_from_prefix(prefix: List[str]) -> str:
    """Extract domain from prefix (first token)."""
    if prefix and prefix[0].startswith('<') and prefix[0].endswith('>'):
        return prefix[0][1:-1]  # Remove < and >
    return 'unknown'


def get_type_from_prefix(prefix: List[str]) -> str:
    """Extract type from argument prefix."""
    if '<type>' in prefix:
        idx = prefix.index('<type>')
        if idx + 1 < len(prefix):
            return prefix[idx + 1]
    return 'unknown'


def format_candidate(candidate: str, domain: str, grounding_type: str, 
                     arg_type: str = None, include_context: bool = True) -> str:
    """
    Format a candidate for the cross-encoder.
    
    Examples:
        - Predicate: "libero_90 predicate : On"
        - Argument: "libero_90 object : moka_pot_1"
    """
    if not include_context:
        return candidate
    
    if grounding_type == 'predicate':
        return f"{domain} predicate : {candidate}"
    else:
        type_str = arg_type if arg_type else 'argument'
        return f"{domain} {type_str} : {candidate}"


def get_sentence_text(entry: Dict) -> str:
    """Get sentence as string."""
    sentence = entry.get('sentence', [])
    if isinstance(sentence, list):
        return ' '.join(sentence)
    return str(sentence)

In [ ]:
class CrossEncoderTrainDataset(Dataset):
    """
    Dataset for training cross-encoder.
    Creates positive and negative pairs from grounding entries.
    """
    
    def __init__(self, data: List[Dict], tokenizer, neg_ratio: int = 3,
                 include_context: bool = True, max_length: int = 128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_context = include_context
        
        # Build training pairs
        self.pairs = []
        
        for entry in data:
            sentence = get_sentence_text(entry)
            target = entry.get('target', [])
            if not target:
                continue
            target = target[0] if isinstance(target, list) else target
            
            candidates = get_candidates_from_prefix(entry)
            if not candidates or target not in candidates:
                continue
            
            domain = get_domain_from_prefix(entry.get('prefix', []))
            grounding_type = entry.get('grounding_type', '')
            arg_type = get_type_from_prefix(entry.get('prefix', [])) if grounding_type != 'predicate' else None
            
            # Positive pair
            pos_candidate = format_candidate(target, domain, grounding_type, arg_type, include_context)
            self.pairs.append((sentence, pos_candidate, 1))
            
            # Negative pairs
            negatives = [c for c in candidates if c != target]
            if negatives:
                n_neg = min(neg_ratio, len(negatives))
                sampled_negs = random.sample(negatives, n_neg)
                for neg in sampled_negs:
                    neg_candidate = format_candidate(neg, domain, grounding_type, arg_type, include_context)
                    self.pairs.append((sentence, neg_candidate, 0))
        
        # Shuffle
        random.shuffle(self.pairs)
        
        print(f"  Created {len(self.pairs)} training pairs ")
        print(f"    Positive: {sum(1 for p in self.pairs if p[2] == 1)}")
        print(f"    Negative: {sum(1 for p in self.pairs if p[2] == 0)}")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        sentence, candidate, label = self.pairs[idx]
        
        encoding = self.tokenizer(
            sentence,
            candidate,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'token_type_ids': encoding.get('token_type_ids', torch.zeros_like(encoding['input_ids'])).squeeze(0),
            'labels': torch.tensor(label, dtype=torch.float),
        }

In [ ]:
def sample_fewshot(data: List[Dict], n: int, seed: int = 42) -> List[Dict]:
    """
    Sample n examples for few-shot training.
    Stratified by grounding_type (predicate vs argument).
    """
    rng = random.Random(seed)
    
    predicates = [e for e in data if e.get('grounding_type') == 'predicate']
    arguments = [e for e in data if e.get('grounding_type', '').startswith('arg_')]
    
    total = len(predicates) + len(arguments)
    if total == 0:
        return rng.sample(data, min(n, len(data)))
    
    pred_ratio = len(predicates) / total
    n_pred = int(n * pred_ratio)
    n_arg = n - n_pred
    
    sampled_pred = rng.sample(predicates, min(n_pred, len(predicates)))
    sampled_arg = rng.sample(arguments, min(n_arg, len(arguments)))
    
    result = sampled_pred + sampled_arg
    rng.shuffle(result)
    
    return result


# Pre-generate few-shot samples
fewshot_samples = {}
for n in FEWSHOT_SIZES:
    if n <= len(libero_train_pool):
        fewshot_samples[n] = sample_fewshot(libero_train_pool, n)
        print(f"Few-shot {n}: {len(fewshot_samples[n])} examples")
    else:
        print(f"[warn] Requested {n} but only {len(libero_train_pool)} available")
        fewshot_samples[n] = libero_train_pool.copy()

## 3. Evaluation Functions

In [ ]:
@torch.no_grad()
def score_candidates(model, tokenizer, sentence: str, candidates: List[str],
                     batch_size: int = 64, max_length: int = 128) -> np.ndarray:
    """
    Score all candidates for a sentence.
    Returns array of scores (higher = more likely match).
    """
    model.eval()
    scores = []
    
    for i in range(0, len(candidates), batch_size):
        batch_candidates = candidates[i:i + batch_size]
        
        encodings = tokenizer(
            [sentence] * len(batch_candidates),
            batch_candidates,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        encodings = {k: v.to(device) for k, v in encodings.items()}
        outputs = model(**encodings)
        batch_scores = torch.sigmoid(outputs['logits']).cpu().numpy()
        scores.extend(batch_scores)
    
    return np.array(scores)


def evaluate_cross_encoder(model, tokenizer, test_data: List[Dict],
                           include_context: bool = True,
                           batch_size: int = 64) -> Dict[str, Any]:
    """
    Evaluate cross-encoder on test data.
    
    For each entry:
    1. Get all candidates from prefix
    2. Score each (sentence, candidate) pair
    3. Check if argmax matches target
    """
    model.eval()
    
    results = []
    correct = 0
    correct_at_3 = 0
    total = 0
    mrr_sum = 0.0
    
    for entry in tqdm(test_data, desc="Evaluating"):
        sentence = get_sentence_text(entry)
        target = entry.get('target', [])
        if not target:
            continue
        target = target[0] if isinstance(target, list) else target
        
        candidates = get_candidates_from_prefix(entry)
        if not candidates or target not in candidates:
            continue
        
        domain = get_domain_from_prefix(entry.get('prefix', []))
        grounding_type = entry.get('grounding_type', '')
        arg_type = get_type_from_prefix(entry.get('prefix', [])) if grounding_type != 'predicate' else None
        
        # Format candidates
        formatted_candidates = [
            format_candidate(c, domain, grounding_type, arg_type, include_context)
            for c in candidates
        ]
        
        # Score
        scores = score_candidates(model, tokenizer, sentence, formatted_candidates, batch_size)
        
        # Get prediction
        pred_idx = int(np.argmax(scores))
        pred_candidate = candidates[pred_idx]
        target_idx = candidates.index(target)
        
        # Metrics
        is_correct = (pred_candidate == target)
        if is_correct:
            correct += 1
        
        # Acc@3
        top3_indices = np.argsort(scores)[-3:][::-1]
        if target_idx in top3_indices:
            correct_at_3 += 1
        
        # MRR
        sorted_indices = np.argsort(scores)[::-1]
        rank = np.where(sorted_indices == target_idx)[0][0] + 1
        mrr_sum += 1.0 / rank
        
        total += 1
        
        results.append({
            'sentence': sentence,
            'target': target,
            'prediction': pred_candidate,
            'correct': is_correct,
            'grounding_type': grounding_type,
            'suite': entry.get('suite', 'unknown'),
            'rank': rank,
        })
    
    # Compute metrics
    accuracy = correct / total if total > 0 else 0
    acc_at_3 = correct_at_3 / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    # Breakdown by grounding type
    pred_results = [r for r in results if r['grounding_type'] == 'predicate']
    arg_results = [r for r in results if r['grounding_type'].startswith('arg_')]
    
    pred_acc = sum(r['correct'] for r in pred_results) / len(pred_results) if pred_results else 0
    arg_acc = sum(r['correct'] for r in arg_results) / len(arg_results) if arg_results else 0
    
    return {
        'overall': {
            'accuracy': accuracy,
            'accuracy_at_3': acc_at_3,
            'mrr': mrr,
            'correct': correct,
            'total': total,
        },
        'predicate': {
            'accuracy': pred_acc,
            'total': len(pred_results),
        },
        'argument': {
            'accuracy': arg_acc,
            'total': len(arg_results),
        },
        'results': results,
    }


def print_results(results: Dict[str, Any], name: str = ""):
    """Pretty print evaluation results."""
    print(f"\n{'='*60}")
    print(f"Results: {name}")
    print(f"{'='*60}")
    
    overall = results.get('overall', {})
    print(f"\nOverall Accuracy: {overall.get('accuracy', 0)*100:.2f}% ({overall.get('correct', 0)}/{overall.get('total', 0)})")
    print(f"  Acc@3: {overall.get('accuracy_at_3', 0)*100:.2f}%")
    print(f"  MRR:   {overall.get('mrr', 0):.4f}")
    
    pred = results.get('predicate', {})
    arg = results.get('argument', {})
    print(f"\nBy Type:")
    print(f"  Predicate: {pred.get('accuracy', 0)*100:.2f}% (n={pred.get('total', 0)})")
    print(f"  Argument:  {arg.get('accuracy', 0)*100:.2f}% (n={arg.get('total', 0)})")

## 4. Training Functions

In [ ]:
def train_cross_encoder(model, tokenizer, train_data: List[Dict],
                        config: Dict, include_context: bool = True) -> Dict:
    """
    Fine-tune a cross-encoder model.
    """
    model.train()
    
    # Create dataset
    dataset = CrossEncoderTrainDataset(
        train_data, tokenizer, 
        neg_ratio=config['neg_ratio'],
        include_context=include_context
    )
    
    dataloader = DataLoader(
        dataset, 
        batch_size=config['batch_size'], 
        shuffle=True
    )
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    total_steps = len(dataloader) * config['num_epochs']
    warmup_steps = int(total_steps * config['warmup_ratio'])
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    criterion = nn.BCEWithLogitsLoss()
    
    # Training loop
    train_losses = []
    
    for epoch in range(config['num_epochs']):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for batch in pbar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            loss = criterion(outputs['logits'], labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        print(f"  Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    
    return {'train_losses': train_losses}

## 5. Run Experiments

In [ ]:
# Load tokenizer and pre-trained model
print("Loading tokenizer and pre-trained model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
pretrained_model = load_pretrained_cross_encoder(PRETRAINED_MODEL_DIR, device)
print(f"Loaded from: {PRETRAINED_MODEL_DIR}")
print(f"Parameters: {sum(p.numel() for p in pretrained_model.parameters()):,}")

In [ ]:
# Zero-shot evaluation
print("\n" + "#" * 60)
print("# Zero-Shot Evaluation (Pre-trained on Original Domains)")
print("#" * 60)

zeroshot_results = evaluate_cross_encoder(
    pretrained_model, tokenizer, libero_test,
    include_context=INCLUDE_CONTEXT,
    batch_size=FINETUNING_CONFIG['eval_batch_size']
)
print_results(zeroshot_results, "Zero-Shot (Pre-trained)")

In [ ]:
# Store all results
all_results = {
    'zero_shot': zeroshot_results,
    'few_shot': {},
    'from_scratch': {},
}

for n_shot in FEWSHOT_SIZES:
    if n_shot not in fewshot_samples:
        continue
    
    train_data = fewshot_samples[n_shot]
    
    print(f"\n{'#'*60}")
    print(f"# Few-Shot Experiment: N = {n_shot}")
    print(f"{'#'*60}")
    
    # --- Few-shot from pre-trained ---
    print(f"\n[1/2] Fine-tuning PRE-TRAINED model on {n_shot} examples...")
    
    # Copy pre-trained model
    fewshot_model = copy_model(pretrained_model, device)
    
    # Train
    train_metrics = train_cross_encoder(
        fewshot_model, tokenizer, train_data,
        config=FINETUNING_CONFIG,
        include_context=INCLUDE_CONTEXT
    )
    
    # Evaluate
    fewshot_results = evaluate_cross_encoder(
        fewshot_model, tokenizer, libero_test,
        include_context=INCLUDE_CONTEXT,
        batch_size=FINETUNING_CONFIG['eval_batch_size']
    )
    all_results['few_shot'][n_shot] = fewshot_results
    print_results(fewshot_results, f"Few-Shot ({n_shot} examples, Pre-trained Init)")
    
    # Save model
    fewshot_dir = OUTPUT_DIR / f"fewshot_{n_shot}"
    fewshot_dir.mkdir(exist_ok=True)
    torch.save(fewshot_model.state_dict(), fewshot_dir / "model.pt")
    
    # Clear memory
    del fewshot_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # --- From scratch ---
    print(f"\n[2/2] Training FROM SCRATCH on {n_shot} examples...")
    
    # Create new model
    scratch_model = create_cross_encoder_from_scratch(device)
    
    # Train
    train_metrics = train_cross_encoder(
        scratch_model, tokenizer, train_data,
        config=FINETUNING_CONFIG,
        include_context=INCLUDE_CONTEXT
    )
    
    # Evaluate
    scratch_results = evaluate_cross_encoder(
        scratch_model, tokenizer, libero_test,
        include_context=INCLUDE_CONTEXT,
        batch_size=FINETUNING_CONFIG['eval_batch_size']
    )
    all_results['from_scratch'][n_shot] = scratch_results
    print_results(scratch_results, f"From Scratch ({n_shot} examples)")
    
    # Save model
    scratch_dir = OUTPUT_DIR / f"scratch_{n_shot}"
    scratch_dir.mkdir(exist_ok=True)
    torch.save(scratch_model.state_dict(), scratch_dir / "model.pt")
    
    # Clear memory
    del scratch_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. Results Summary

In [ ]:
# Build summary table
summary_rows = []

# Zero-shot
zs = all_results['zero_shot']
summary_rows.append({
    'Setting': 'Zero-Shot (Pre-trained)',
    'N': 0,
    'Overall Acc': zs['overall']['accuracy'] * 100,
    'Acc@3': zs['overall']['accuracy_at_3'] * 100,
    'MRR': zs['overall']['mrr'],
    'Predicate Acc': zs['predicate']['accuracy'] * 100,
    'Argument Acc': zs['argument']['accuracy'] * 100,
})

# Few-shot and from-scratch
for n in FEWSHOT_SIZES:
    if n in all_results['few_shot']:
        fs = all_results['few_shot'][n]
        summary_rows.append({
            'Setting': 'Few-Shot (Pre-trained)',
            'N': n,
            'Overall Acc': fs['overall']['accuracy'] * 100,
            'Acc@3': fs['overall']['accuracy_at_3'] * 100,
            'MRR': fs['overall']['mrr'],
            'Predicate Acc': fs['predicate']['accuracy'] * 100,
            'Argument Acc': fs['argument']['accuracy'] * 100,
        })
    
    if n in all_results['from_scratch']:
        sc = all_results['from_scratch'][n]
        summary_rows.append({
            'Setting': 'From Scratch',
            'N': n,
            'Overall Acc': sc['overall']['accuracy'] * 100,
            'Acc@3': sc['overall']['accuracy_at_3'] * 100,
            'MRR': sc['overall']['mrr'],
            'Predicate Acc': sc['predicate']['accuracy'] * 100,
            'Argument Acc': sc['argument']['accuracy'] * 100,
        })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "="*100)
print("SUMMARY: Cross-Encoder Few-Shot Generalization Experiment")
print("="*100)
print(summary_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# Compute transfer benefit
print("\n" + "="*80)
print("TRANSFER BENEFIT ANALYSIS")
print("(Positive = Pre-training helps)")
print("="*80)

print(f"\n{'N':>5} | {'Few-Shot':>12} | {'Scratch':>12} | {'Benefit':>12} | {'Relative':>12}")
print("-" * 60)

for n in FEWSHOT_SIZES:
    if n in all_results['few_shot'] and n in all_results['from_scratch']:
        fs_acc = all_results['few_shot'][n]['overall']['accuracy'] * 100
        sc_acc = all_results['from_scratch'][n]['overall']['accuracy'] * 100
        benefit = fs_acc - sc_acc
        relative = (fs_acc / sc_acc - 1) * 100 if sc_acc > 0 else float('inf')
        
        print(f"{n:>5} | {fs_acc:>10.1f}% | {sc_acc:>10.1f}% | {benefit:>+10.1f}% | {relative:>+10.1f}%")

In [ ]:
# Save results
def convert_for_json(obj):
    if isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items() if k != 'results'}
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (list, tuple)):
        return [convert_for_json(x) for x in obj]
    return obj

results_file = OUTPUT_DIR / "experiment_results.json"
with open(results_file, 'w') as f:
    json.dump(convert_for_json(all_results), f, indent=2)
print(f"Results saved to: {results_file}")

summary_file = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"Summary saved to: {summary_file}")

In [ ]:
# Generate LaTeX table
print("\n" + "="*80)
print("LATEX TABLE FOR PAPER")
print("="*80)

latex = r"""
\begin{table}[h]
    \centering
    \caption{Cross-encoder few-shot generalization to LIBERO. Pre-training on original domains enables strong zero-shot transfer and efficient adaptation.}
    \begin{tabular}{lcccc}
        \toprule
        Setting & N & Accuracy & Acc@3 & MRR \\
        \midrule
"""

# Zero-shot
zs = all_results['zero_shot']
latex += f"        Zero-Shot & 0 & {zs['overall']['accuracy']*100:.1f}\\% & {zs['overall']['accuracy_at_3']*100:.1f}\\% & {zs['overall']['mrr']:.3f} \\\\\n"
latex += "        \\midrule\n"

# Few-shot vs scratch
for n in FEWSHOT_SIZES:
    if n in all_results['few_shot'] and n in all_results['from_scratch']:
        fs = all_results['few_shot'][n]
        sc = all_results['from_scratch'][n]
        
        latex += f"        Few-Shot (Pre-trained) & {n} & {fs['overall']['accuracy']*100:.1f}\\% & {fs['overall']['accuracy_at_3']*100:.1f}\\% & {fs['overall']['mrr']:.3f} \\\\\n"
        latex += f"        From Scratch & {n} & {sc['overall']['accuracy']*100:.1f}\\% & {sc['overall']['accuracy_at_3']*100:.1f}\\% & {sc['overall']['mrr']:.3f} \\\\\n"
        if n != FEWSHOT_SIZES[-1]:
            latex += "        \\hdashline\n"

latex += r"""        \bottomrule
    \end{tabular}
    \label{tab:crossencoder_fewshot}
\end{table}
"""

print(latex)

latex_file = OUTPUT_DIR / "fewshot_table.tex"
with open(latex_file, 'w') as f:
    f.write(latex)
print(f"\nLaTeX saved to: {latex_file}")

## 7. Interpretation Guide

### Expected Results

| Metric | Zero-Shot | Few-Shot (50) | Few-Shot (100) |
|--------|-----------|---------------|----------------|
| Overall Acc | ~42% | ~55-65%? | ~65-75%? |
| Acc@3 | ~71% | ~80-85%? | ~85-90%? |

### Key Claims for Paper

1. **Cross-encoder enables transfer**: 42% zero-shot (vs 9.5% for prefix model)
2. **Few-shot closes the gap**: With N examples, accuracy improves to X%
3. **Pre-training is valuable**: Few-shot consistently beats from-scratch
4. **Practical deployment**: Acc@3 of 71%+ means correct answer usually in top candidates

### Paper Sentence

> "Our cross-encoder architecture achieves 42% zero-shot accuracy on unseen LIBERO 
> signatures—a 4.4× improvement over position-based classification. With only 50 
> examples of fine-tuning, accuracy improves to X%, demonstrating efficient 
> adaptation to novel system vocabularies."